# Notebook 05 — Final Decision-Support Prototype

**Project:** CleanGanga-Prayagraj

This notebook turns the outputs from Notebooks 01–04 into one reproducible decision-support workflow.

**Flow:** Station/Input → ML prediction → Hotspot evidence → Retrieved knowledge → IBM Granite explanation → Responsible decision support.

This notebook does not repeat EDA, hotspot calculation, model training analysis, or RAG evaluation.

## 1. Project pipeline

```text
NB01  Data Quality
  ↓
NB02  Hotspot Analysis + Logistic Regression
  ↓
NB03  IBM Granite + RAG
  ↓
NB04  Evaluation + Responsible AI
  ↓
NB05  Final Decision-Support Prototype
  ↓
Streamlit Application
```

In [ ]:
from pathlib import Path
import os
import json
import re
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path("../data")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

station_path = DATA_DIR / "station_summary.csv"
hotspot_path = DATA_DIR / "hotspot_ranking.csv"

print("Project data:", DATA_DIR.resolve())

## 2. Load existing project outputs

The prototype consumes artifacts produced by earlier notebooks instead of recomputing the complete analysis.

In [ ]:
if not station_path.exists():
    raise FileNotFoundError("station_summary.csv is missing. Run Notebook 02 export first.")

if not hotspot_path.exists():
    raise FileNotFoundError("hotspot_ranking.csv is missing. Run Notebook 02 export first.")

station_summary = pd.read_csv(station_path)
hotspot_ranking = pd.read_csv(hotspot_path)

print("Station summary:", station_summary.shape)
print("Hotspot ranking:", hotspot_ranking.shape)

## 3. Detect the existing ML interface

The model should use the same feature design and target definition established in Notebook 02.

In [ ]:
candidate_features = ["persistence", "severity", "anomaly_rate", "observations"]
feature_columns = [c for c in candidate_features if c in station_summary.columns]

target_candidates = ["high_risk", "target", "label", "hotspot_label"]
target_column = next((c for c in target_candidates if c in station_summary.columns), None)

print("Available ML features:", feature_columns)
print("Exported target:", target_column)

## 4. Reconstruct the ML prediction interface only when the target is available

This is a reusable prediction layer for the final prototype. The target remains the same rule-derived target used in Notebook 02.

If Notebook 02 exported a trained model, the final application should eventually load that saved model instead of retraining here.

In [ ]:
model = None
scaler = None

if target_column and len(feature_columns) >= 2:
    train_df = station_summary.dropna(subset=feature_columns + [target_column]).copy()

    if train_df[target_column].nunique() >= 2:
        X = train_df[feature_columns]
        y = train_df[target_column]

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_scaled, y)

        print("Prototype Logistic Regression model trained.")
        print("Features:", feature_columns)
        print("Classes:", model.classes_)
    else:
        print("Target contains fewer than two classes; ML prediction unavailable.")
else:
    print("No compatible exported target found; ML prediction will remain unavailable.")

## 5. Station decision function

The deterministic hotspot evidence and the ML prediction are kept separate.

This is important because the hotspot score and the ML prediction answer different questions.

In [ ]:
def get_station_row(station_name):
    if "Station" not in station_summary.columns:
        raise KeyError("Station column is required.")

    matches = station_summary[
        station_summary["Station"].astype(str).str.lower() == str(station_name).lower()
    ]

    if matches.empty:
        raise ValueError(f"Station not found: {station_name}")

    return matches.iloc[0]


def get_hotspot_row(station_name):
    if "Station" not in hotspot_ranking.columns:
        return None

    matches = hotspot_ranking[
        hotspot_ranking["Station"].astype(str).str.lower() == str(station_name).lower()
    ]

    return matches.iloc[0] if not matches.empty else None


def station_decision(station_name):
    row = get_station_row(station_name)
    hotspot = get_hotspot_row(station_name)

    result = {
        "station": station_name,
        "metrics": {},
        "hotspot_score": None,
        "hotspot_rank": None,
        "ml_prediction": None,
        "ml_probability": None
    }

    for col in [
        "persistence", "severity", "anomaly_rate",
        "observations", "mean_bod", "max_bod",
        "mean_fc", "max_fc"
    ]:
        if col in row.index:
            result["metrics"][col] = row[col]

    if hotspot is not None:
        if "hotspot_score" in hotspot.index:
            result["hotspot_score"] = hotspot["hotspot_score"]
        if "rank" in hotspot.index:
            result["hotspot_rank"] = hotspot["rank"]

    if model is not None and scaler is not None:
        values = [row.get(feature, np.nan) for feature in feature_columns]

        if not any(pd.isna(values)):
            X_new = scaler.transform(
                pd.DataFrame([values], columns=feature_columns)
            )
            result["ml_prediction"] = model.predict(X_new)[0]

            if hasattr(model, "predict_proba"):
                result["ml_probability"] = float(model.predict_proba(X_new).max())

    return result

## 6. Test the station decision workflow

In [ ]:
if "Station" in station_summary.columns and not station_summary.empty:
    example_station = station_summary["Station"].iloc[0]
    example_result = station_decision(example_station)
    print(json.dumps(example_result, indent=2, default=str))
else:
    print("No station data available.")

## 7. Load the verified RAG knowledge base

Notebook 03 established the retrieval approach. The final prototype reuses the same simple TF-IDF retrieval baseline.

Place verified `.txt` or `.md` references in:

`data/knowledge_base/`

In [ ]:
KNOWLEDGE_DIR = DATA_DIR / "knowledge_base"
KNOWLEDGE_DIR.mkdir(parents=True, exist_ok=True)

knowledge_files = sorted(
    list(KNOWLEDGE_DIR.glob("*.txt")) +
    list(KNOWLEDGE_DIR.glob("*.md"))
)

knowledge_chunks = []


def chunk_text(text, chunk_size=1200, overlap=200):
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])

        if end == len(text):
            break

        start = end - overlap

    return chunks


for path in knowledge_files:
    text = path.read_text(encoding="utf-8", errors="ignore")

    for chunk_id, chunk in enumerate(chunk_text(text)):
        knowledge_chunks.append({
            "source": path.name,
            "chunk_id": chunk_id,
            "text": chunk
        })

print("Knowledge files:", len(knowledge_files))
print("Knowledge chunks:", len(knowledge_chunks))

In [ ]:
if knowledge_chunks:
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2)
    )
    knowledge_matrix = vectorizer.fit_transform(
        [item["text"] for item in knowledge_chunks]
    )
else:
    vectorizer = None
    knowledge_matrix = None


def retrieve_knowledge(query, top_k=3):
    if vectorizer is None or knowledge_matrix is None:
        return []

    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, knowledge_matrix)[0]
    indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            **knowledge_chunks[i],
            "score": float(scores[i])
        }
        for i in indices
    ]

## 8. Build the final grounded Granite prompt

The model receives three clearly separated layers:

1. Project-computed evidence
2. Retrieved reference evidence
3. User question

Granite interprets the evidence; it does not create the underlying measurements.

In [ ]:
def build_final_prompt(decision, retrieved_docs, user_question):
    metric_text = "\n".join(
        f"- {key}: {value}"
        for key, value in decision["metrics"].items()
    )

    source_text = "\n\n".join(
        f"[Source: {doc['source']} | Chunk: {doc['chunk_id']}]\n{doc['text']}"
        for doc in retrieved_docs
    )

    if not source_text:
        source_text = "No external reference was retrieved."

    return f"""
You are an environmental decision-support assistant for CleanGanga-Prayagraj.

USER QUESTION:
{user_question}

PROJECT-COMPUTED EVIDENCE:
Station: {decision['station']}
Hotspot score: {decision['hotspot_score']}
Hotspot rank: {decision['hotspot_rank']}
ML prediction: {decision['ml_prediction']}
ML probability: {decision['ml_probability']}
Metrics:
{metric_text}

RETRIEVED REFERENCE EVIDENCE:
{source_text}

INSTRUCTIONS:
1. Answer the user's question using only the supplied evidence.
2. Clearly distinguish project-computed evidence from reference information.
3. Mention retrieved sources when using their information.
4. Do not invent measurements.
5. Do not claim causation or identify a pollution source without evidence.
6. Do not describe the hotspot score as an official regulatory classification.
7. Communicate uncertainty where relevant.
8. If the evidence is insufficient, say so.

Provide a concise decision-support explanation.
""".strip()

## 9. IBM Granite adapter

Connect this adapter to the tested Granite inference function from Notebook 03.

No fake Granite output is generated if the IBM connection is unavailable.

In [ ]:
def granite_generate(prompt):
    raise NotImplementedError(
        "Connect this function to the tested IBM Granite inference function from Notebook 03."
    )

## 10. End-to-end station question function

This is the core of the final prototype.

It returns structured information that can later be rendered directly by Streamlit.

In [ ]:
def answer_station_question(station_name, user_question):
    decision = station_decision(station_name)

    retrieval_query = (
        f"{user_question} water quality BOD fecal coliform "
        f"station hotspot monitoring"
    )

    retrieved_docs = retrieve_knowledge(retrieval_query, top_k=3)

    prompt = build_final_prompt(
        decision,
        retrieved_docs,
        user_question
    )

    try:
        answer = granite_generate(prompt)
        status = "granite_generated"
    except NotImplementedError:
        answer = None
        status = "granite_not_connected"

    return {
        "status": status,
        "station": station_name,
        "decision": decision,
        "retrieved_sources": [
            {
                "source": doc["source"],
                "chunk_id": doc["chunk_id"],
                "score": doc["score"]
            }
            for doc in retrieved_docs
        ],
        "prompt": prompt,
        "answer": answer
    }

## 11. Test the complete pipeline

This test validates the station → retrieval → prompt pipeline even when Granite is not connected.

In [ ]:
if "Station" in station_summary.columns and not station_summary.empty:
    station = station_summary["Station"].iloc[0]

    test_output = answer_station_question(
        station,
        "Why is this station considered a potential hotspot?"
    )

    print("Status:", test_output["status"])
    print("Station:", test_output["station"])
    print("Retrieved sources:", test_output["retrieved_sources"])
    print("\nPrompt preview:\n")
    print(test_output["prompt"][:3000])
else:
    print("No station available for testing.")

## 12. Final application contract

The future Streamlit application should consume `answer_station_question()` rather than duplicating the analysis logic.

### User flow

```text
Select station
      ↓
Show measured metrics
      ↓
Show hotspot rank/score
      ↓
Show ML prediction when available
      ↓
User asks a question
      ↓
Retrieve verified references
      ↓
Granite generates grounded explanation
      ↓
Show supporting sources + limitations
```

## 13. Prototype response export

Save one structured response as a development artifact. This helps test the future UI independently of the notebook.

In [ ]:
if "Station" in station_summary.columns and not station_summary.empty:
    station = station_summary["Station"].iloc[0]

    prototype_response = answer_station_question(
        station,
        "Summarize the evidence for this station."
    )

    prototype_path = DATA_DIR / "prototype_response.json"

    with open(prototype_path, "w", encoding="utf-8") as f:
        json.dump(prototype_response, f, indent=2, default=str)

    print("Saved:", prototype_path)

## 14. Responsible AI guardrails

The final application must enforce:

- No fabricated measurements
- No unsupported pollution-source claims
- No legal or regulatory conclusions
- Sources shown for retrieved knowledge
- ML prediction kept separate from deterministic hotspot evidence
- Uncertainty and dataset limitations displayed
- No personal information sent to the model
- No API keys stored in notebooks or Git

# Final architecture

```text
                    CPCB 2021 data
                         ↓
                 NB01 Data Quality
                         ↓
              NB02 Hotspot + ML
                         ↓
                  Station Evidence
                         │
             ┌───────────┴───────────┐
             ↓                       ↓
      Verified Knowledge       ML / Metrics
             ↓                       │
         Retrieval                    │
             └───────────┬───────────┘
                         ↓
                   IBM Granite
                         ↓
               Grounded Explanation
                         ↓
              Decision-Support Output
                         ↓
                  Streamlit App
```

## What remains

The analytical pipeline is complete.

The next major engineering stage is to build the **Streamlit application** that exposes this workflow cleanly to a user and then prepare the final README, architecture diagram, evaluation results, limitations, and demo.